# 라이브러리 불러오기

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import torchvision.transforms as transforms
import optuna
import numpy as np

# 데이터 평균 표준편차 구하기

In [ ]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class FlatImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
        self.files = [f for f in os.listdir(root) if f.lower().endswith(exts)]
        if len(self.files) == 0:
            raise FileNotFoundError(f"No images found in: {root}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = os.path.join(self.root, self.files[idx])
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

root = r"C:\project\TEAM-PJ-DEEP\복만수\data1\train_4000"
dataset = FlatImageDataset(root, transform=transform)
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)

# ✅ 전체 픽셀 기준 누적합/제곱합
channel_sum = torch.zeros(3)
channel_sq_sum = torch.zeros(3)
num_pixels = 0

for images in loader:  # images: (B,3,224,224)
    b, c, h, w = images.shape
    num_pixels += b * h * w
    channel_sum += images.sum(dim=[0, 2, 3])
    channel_sq_sum += (images ** 2).sum(dim=[0, 2, 3])

mean = channel_sum / num_pixels
var = (channel_sq_sum / num_pixels) - (mean ** 2)
std = torch.sqrt(var)

print("mean:", mean)
print("std:", std)

# 정규화

In [ ]:
from torchvision import transforms

MEAN = [0.9367, 0.9364, 0.9358]
STD  = [0.0957, 0.0964, 0.0963]

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# 데이터 불러오기
## 이미지와 라벨(csv)을 함께 읽어서 학습용 데이터셋으로 만드는 코드

In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd
from PIL import Image
import os

class MultiLabelDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.labels = self.data.iloc[:, 1:].values

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.data.iloc[idx, 0])
        image = Image.open(img_path).convert("RGB")
        label = torch.tensor(self.labels[idx], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label

# 데이터를 학습용/검증용/테스트용 Dataset 객체 생성

In [ ]:
num_classes = 14

train_dataset = MultiLabelDataset(
    csv_file="./data1/train_labels_4000.csv",
    img_dir="./data1/train_4000",
    transform=transform_train
)

val_dataset = MultiLabelDataset(
    csv_file="./data1/val_labels_4000.csv",
    img_dir="./data1/val_4000",
    transform=transform_val
)

test_dataset = MultiLabelDataset(
    csv_file="./data1/test_labels_4000.csv",
    img_dir="./data1/test_4000",
    transform=transform_test
)

# DataLoader 생성
## Dataset을 실제 학습에 사용할 수 있게 배치 단위로 묶는 단계

In [ ]:
from torch.utils.data import DataLoader

def get_train_val_loaders(batch_size):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=0)
    # test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader, # test_loader

# 랜덤 시드 고정

In [ ]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
SEED = 42
set_seed(SEED)

def get_train_val_loaders(train_dataset, val_dataset, batch_size):
    g = torch.Generator()
    g.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        generator=g
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    return train_loader, val_loader

# 쿠다 사용확인

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("현재 device:", device)

if torch.cuda.is_available():
    print("GPU 개수:", torch.cuda.device_count())
    print("지정한 GPU 이름:", torch.cuda.get_device_name(1))

# 클래스 불균형 보정 pos_weight 계산 단계

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

df = pd.read_csv("./data1/train_labels_4000.csv")
labels = df.iloc[:, 1:].values.astype(np.float32)

pos_counts = labels.sum(axis=0)
neg_counts = labels.shape[0] - pos_counts

# 0 방지
pos_counts = np.clip(pos_counts, 1.0, None)

pos_weight = neg_counts / pos_counts
pos_weight = torch.tensor(pos_weight, dtype=torch.float32).to(device)
# pos_weight는 양성 클래스(값이 1인 경우)에 더 큰 중요도를 주는 가중치
# 1 = 해당 항목이 작성됨 0 = 해당 항목이 비어있음 일 때
# 실제 데이터는 많은 경우 0이 훨씬 만다 
# 이럴 때 모델이 대충 다 0이라고만 예측해도 손실이 크게 안나서 1을 잘 못잡는 문제가 생긴다
# pos_weight를 써서 1을 틀렸을 떄는 더 크게 벌주고, 0을 틀렸을 때보다 더 중요하게 보게 만드는 것

# Optuna 일 때 모델 불러오기

def create_model(num_classes, device, dropout_rate=0.0):
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

    # 전체 freeze
    for p in model.parameters():
        p.requires_grad = False

    in_features = model.fc.in_features

    # fc 교체
    if dropout_rate > 0:
        model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, num_classes)
        )
    else:
        model.fc = nn.Linear(in_features, num_classes)

    # layer4 unfrozen
    for p in model.layer4.parameters():
        p.requires_grad = True

    # fc unfrozen
    for p in model.fc.parameters():
        p.requires_grad = True

    model = model.to(device)
    return model

model = create_model(
    num_classes=num_classes,
    device=device,
    dropout_rate=0.0
)

# 모델 확인

model

# 지금 어떤게 안얼었는지 얼었는지 확인

In [ ]:
# 지금 어떤게 안얼었는지 얼었는지 확인
for name,module in model.named_parameters():
    print(name , module.requires_grad)

# 학습하기(Optuna)

In [ ]:
import os
import copy
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import tqdm

from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import models

device = "cuda:1" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "resnet50"
DATA_TAG = "4000"

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("runs", exist_ok=True)

BEST_MODEL_PATH = f"checkpoints/best_model_{MODEL_NAME}_{DATA_TAG}_optuna_1.pth"

def objective(trial):
    # ---------------------------
    # 1) 하이퍼파라미터 탐색 공간
    # ---------------------------
    layer4_lr = trial.suggest_float("layer4_lr", 1e-5, 3e-4, log=True)
    fc_lr = trial.suggest_float("fc_lr", 1e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])

    # ---------------------------
    # 2) DataLoader 생성
    # ---------------------------
    train_loader, val_loader = get_train_val_loaders(train_dataset, val_dataset, batch_size)

    # ---------------------------
    # 3) 모델 / 손실 / 옵티마이저
    # ---------------------------
    model = create_model(
        num_classes=num_classes,
        device=device,
    )

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

    if optimizer_name == "Adam":
        optimizer = optim.Adam(
            [
                {"params": model.layer4.parameters(), "lr": layer4_lr},
                {"params": model.fc.parameters(), "lr": fc_lr},
            ]
        )
    else:
        optimizer = optim.SGD(
            [
                {"params": model.layer4.parameters(), "lr": layer4_lr},
                {"params": model.fc.parameters(), "lr": fc_lr},
            ],
            momentum=0.9
        )

    # ---------------------------
    # 4) Checkpoint 경로
    # ---------------------------
    trial_ckpt_path = os.path.join(
        "checkpoints", f"best_model_{DATA_TAG}_trial_{trial.number}.pth"
    )

    # ---------------------------
    # 5) 학습 설정
    # ---------------------------
    EPOCHS = 30
    stop_count = 5
    early_stop_count = 0
    best_val_loss = float("inf")

    train_epoch_losses = []
    val_epoch_losses = []
    train_batch_losses = []   # ✅ 배치 loss 저장용

    # ---------------------------
    # 6) 학습 루프
    # ---------------------------
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        batch_count = 0

        for img, labels in tqdm.tqdm(train_loader, desc=f"Trial {trial.number} | Epoch {epoch+1}"):
            img = img.to(device)
            labels = labels.to(device).float()

            optimizer.zero_grad()
            preds = model(img)
            loss = criterion(preds, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            batch_count += 1

            train_batch_losses.append(loss.item())   # ✅ batch loss 저장

        avg_train_loss = running_loss / max(batch_count, 1)

        # ---------------------------
        # 7) Validation
        # ---------------------------
        model.eval()
        val_loss_sum = 0.0
        val_batch_count = 0

        with torch.no_grad():
            for img, labels in val_loader:
                img = img.to(device)
                labels = labels.to(device).float()

                pred = model(img)
                loss = criterion(pred, labels)

                val_loss_sum += loss.item()
                val_batch_count += 1

        total_val_loss = val_loss_sum / max(val_batch_count, 1)

        train_epoch_losses.append(avg_train_loss)
        val_epoch_losses.append(total_val_loss)

        # ---------------------------
        # 8) Optuna pruning
        # ---------------------------
        trial.report(total_val_loss, epoch)

        if trial.should_prune():
            trial.set_user_attr("train_epoch_losses", train_epoch_losses)
            trial.set_user_attr("val_epoch_losses", val_epoch_losses)
            trial.set_user_attr("train_batch_losses", train_batch_losses)  # ✅
            raise optuna.exceptions.TrialPruned()

        # ---------------------------
        # 9) Checkpoint 저장
        # ---------------------------
        if total_val_loss < best_val_loss:
            early_stop_count = 0
            best_val_loss = total_val_loss
            best_state_dict = copy.deepcopy(model.state_dict())

            torch.save(
                {
                    "model_state": best_state_dict,
                    "trial_number": trial.number,
                    "best_val_loss": best_val_loss,
                    "params": trial.params,
                },
                trial_ckpt_path
            )
        else:
            early_stop_count += 1
            if early_stop_count >= stop_count:
                print(f"🛑 Early stopping | Trial {trial.number}")
                break

        print(
            f"[Trial {trial.number}] "
            f"Epoch {epoch+1} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {total_val_loss:.4f} | "
            f"Best Val Loss: {best_val_loss:.4f} | "
            f"EarlyStopCount: {early_stop_count}"
        )

    # ---------------------------
    # 10) trial 정보 기록
    # ---------------------------
    trial.set_user_attr("best_val_loss", best_val_loss)
    trial.set_user_attr("checkpoint_path", trial_ckpt_path)
    trial.set_user_attr("train_epoch_losses", train_epoch_losses)
    trial.set_user_attr("val_epoch_losses", val_epoch_losses)
    trial.set_user_attr("train_batch_losses", train_batch_losses)   # ✅

    return best_val_loss


sampler = optuna.samplers.TPESampler(seed=SEED)

study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=3,
        n_warmup_steps=3,
        interval_steps=1
    )
)

study.optimize(objective, n_trials=20)

print("Best Trial:")
print("  Trial Number:", study.best_trial.number)
print("  Value (best val loss):", study.best_trial.value)
print("  Checkpoint:", study.best_trial.user_attrs["checkpoint_path"])

print("\n===== BEST HYPERPARAMETERS =====")
for key, value in study.best_trial.params.items():
    print(f"{key}: {value}")

best_ckpt_path = study.best_trial.user_attrs["checkpoint_path"]
best_ckpt = torch.load(best_ckpt_path, map_location=device)

torch.save(best_ckpt, BEST_MODEL_PATH)

print("최종 best 모델 저장:", BEST_MODEL_PATH)

best_trial_number = study.best_trial.number
best_tb_dir = os.path.join("runs", f"best_trial_{DATA_TAG}_{best_trial_number}_1")

writer = SummaryWriter(log_dir=best_tb_dir)

best_train_losses = study.best_trial.user_attrs["train_epoch_losses"]
best_val_losses = study.best_trial.user_attrs["val_epoch_losses"]
best_train_batch_losses = study.best_trial.user_attrs["train_batch_losses"]   # ✅

# ✅ best trial의 batch loss만 저장
for step, loss in enumerate(best_train_batch_losses):
    writer.add_scalar("Loss/train", loss, step)

# 기존 epoch 그래프도 같이 저장
for epoch, loss in enumerate(best_train_losses):
    writer.add_scalar("Loss/train_epoch", loss, epoch)

for epoch, loss in enumerate(best_val_losses):
    writer.add_scalar("Loss/val_epoch", loss, epoch)

writer.add_text("Best Trial Info", f"Trial Number: {best_trial_number}", 0)
writer.add_text("Best Val Loss", f"{study.best_trial.value:.6f}", 0)

hp_text = "\n".join([f"{k}: {v}" for k, v in study.best_trial.params.items()])
writer.add_text("Hyperparameters", hp_text, 0)

writer.close()

print("최종 TensorBoard 저장:", best_tb_dir)

# test 폴더 전체 확인 값

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
THRESHOLD = 0.5
EPS = 1e-12

model = model.to(device)
model.eval()

# accuracy용
total_labels = 0
correct_labels = 0
total_samples = 0
correct_samples = 0

# F1용
tp_micro = 0
fp_micro = 0
fn_micro = 0

num_classes = 14
tp_c = torch.zeros(num_classes, dtype=torch.float64)
fp_c = torch.zeros(num_classes, dtype=torch.float64)
fn_c = torch.zeros(num_classes, dtype=torch.float64)

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        labels = labels.to(device).int()

        logits = model(imgs)
        probs = torch.sigmoid(logits)
        preds = (probs > THRESHOLD).int()

        # acc
        correct_labels += (preds == labels).sum().item()
        total_labels += labels.numel()

        correct_samples += (preds == labels).all(dim=1).sum().item()
        total_samples += labels.size(0)

        # micro F1
        tp_micro += ((preds == 1) & (labels == 1)).sum().item()
        fp_micro += ((preds == 1) & (labels == 0)).sum().item()
        fn_micro += ((preds == 0) & (labels == 1)).sum().item()

        # macro F1
        tp_c += ((preds == 1) & (labels == 1)).sum(dim=0).cpu().double()
        fp_c += ((preds == 1) & (labels == 0)).sum(dim=0).cpu().double()
        fn_c += ((preds == 0) & (labels == 1)).sum(dim=0).cpu().double()

# accuracy
label_acc = correct_labels / total_labels
subset_acc = correct_samples / total_samples

# micro F1
precision_micro = tp_micro / (tp_micro + fp_micro + EPS)
recall_micro = tp_micro / (tp_micro + fn_micro + EPS)
f1_micro = 2 * precision_micro * recall_micro / (precision_micro + recall_micro + EPS)

# macro F1
precision_per_class = tp_c / (tp_c + fp_c + EPS)
recall_per_class = tp_c / (tp_c + fn_c + EPS)
f1_per_class = 2 * precision_per_class * recall_per_class / (precision_per_class + recall_per_class + EPS)
f1_macro = f1_per_class.mean().item()

print(f"label_acc:  {label_acc*100:.2f}% (모든 테스트 이미지의 모든 라벨을 다 펼처서 정확도 확인)")
print(f"subset_acc: {subset_acc*100:.2f}% (14개 전부 맞춘 비율)")
print(f"f1_micro:   {f1_micro*100:.2f}% (작성됨(1)에 대해 잘못 예측한 경우(FP)와 놓친 경우(FN)를 함께 고려한 전체 성능 지표)")
print(f"f1_macro:   {f1_macro*100:.2f}% (라벨별 F1을 구해 평균 14개 항목을 비슷하게 잘 맞추고 있다는 신호)")

# 딱 1장에 대해서 test

import os
import pandas as pd
import torch
from PIL import Image
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
THRESHOLD = 0.5

img_name = "학습-손글씨체-전입신고서3.jpg"
img_path = os.path.join("./data/test", img_name)

# 1) 이미지
image = Image.open(img_path).convert("RGB")
x = transform_test(image).unsqueeze(0).to(device)

# 2) 정답 라벨 찾기
df = pd.read_csv("./data1/test_labels_4000.csv")

row = df[df.iloc[:, 0].astype(str).str.strip() == img_name]
if len(row) == 0:
    raise ValueError(f"{img_name} not found in test_labels.csv")

# ✅ 라벨을 숫자로 강제 변환 (문자/빈칸은 0으로 처리)
label_series = row.iloc[0, 1:]
label_numeric = pd.to_numeric(label_series, errors="coerce").fillna(0).astype(np.int64)

y = torch.tensor(label_numeric.values, dtype=torch.int64, device=device)  # (14,)

# 3) 추론
model = model.to(device)
model.eval()
with torch.no_grad():
    logits = model(x)                         # (1,14)
    probs = torch.sigmoid(logits).squeeze(0)  # (14,)
    pred = (probs > THRESHOLD).int()          # (14,)

# 4) 1장 label_acc / subset_acc
correct_labels = (pred == y).sum().item()
total_labels = y.numel()
label_acc = correct_labels / total_labels
subset_acc = 1.0 if (pred == y).all().item() else 0.0

print("GT  :", y.cpu().tolist(), "test_labels.csv에서 가져온 값")
print("PRED:", pred.cpu().tolist(), "모델이 예측한 값")
print("PROB:", [round(v, 3) for v in probs.cpu().tolist()])
print(f"label_acc:  {label_acc*100:.2f}%")
print(f"subset_acc: {subset_acc*100:.2f}%  (14개 전부 맞춘 비율)")

# Grad-CAM

# 라이브러리 불러오기

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# 모델 불러오기

In [ ]:
checkpoint = torch.load('./checkpoints/best_model_resnet50_1000_optuna.pth', map_location=device)

dropout_rate = checkpoint["params"].get("dropout", 0.0)

model = create_model(
    num_classes=num_classes,
    device=device,
    dropout_rate=dropout_rate
)

model.load_state_dict(checkpoint["model_state"])
model = model.to(device)
model.eval()

# 추론

In [ ]:
from PIL import Image
import torch

model.eval()
model.to('cuda')

img = Image.open('data/test/학습-전입신고서436.jpg').convert('RGB')
input_tensor = transform_test(img).unsqueeze(0).to('cuda')

with torch.no_grad():
    pred = model(input_tensor)          # logits
    probs = torch.sigmoid(pred)         # 각 라벨 확률
    preds = (probs > 0.5).int()         # threshold 0.5 기준 0/1

print("logits:", pred)
print("probs:", probs)
print("preds:", preds)

# 타겟레이어 설정

In [ ]:
target_layers = [model.layer4[-1]]
# 마지막 레이어를 가져오는게 좋은데 모르겠으면 지피티한테 물어보기 
# 마지막 에이어가 몇번이야?

# 최종 Grad- CAM 확인

In [ ]:
if input_tensor.dim() == 3:
    input_tensor = input_tensor.unsqueeze(0)

cam_input = input_tensor.clone().detach().float().to(device)
cam_input.requires_grad_(True)

cam = GradCAM(model=model, target_layers=target_layers)

target_label = 10   # 보고 싶은 클래스 번호

grayscale_cam = cam(
    input_tensor=cam_input,
    targets=[ClassifierOutputTarget(target_label)]
)

grad_cam = grayscale_cam[0]

rgb_img = np.array(img.resize((224, 224))).astype(np.float32) / 255.0
visualization = show_cam_on_image(rgb_img, grad_cam, use_rgb=True)

plt.figure(figsize=(6, 6))
plt.imshow(visualization)
plt.axis("off")
plt.show()